# EDA Story: Telecom Churn Risk

This notebook focuses on business-relevant insights, not only charts.

## Objective
Identify behavioral and contractual patterns associated with churn risk and translate them into actionable retention strategy.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from generate_messy_data import SimulationConfig, simulate_messy_data

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

cfg = SimulationConfig(n_customers=2500, seed=7, reference_date="2024-12-31")
_, _, _, abt_df = simulate_messy_data(cfg)

print("ABT shape:", abt_df.shape)
abt_df[["churned", "region", "login_count", "days_since_last_login", "total_amount"]].head()

## 1) Class Imbalance Check

Before modeling, we verify class balance because churn datasets are often skewed.

In [ ]:
churn_dist = abt_df["churned"].value_counts(normalize=True).rename("ratio").reset_index()
churn_dist.columns = ["churned", "ratio"]

plt.figure(figsize=(6, 4))
sns.barplot(data=churn_dist, x="churned", y="ratio", hue="churned", dodge=False, legend=False)
plt.title("Churn Class Distribution")
plt.xlabel("Churned")
plt.ylabel("Ratio")
plt.tight_layout()
plt.show()

print(churn_dist)

### Takeaway

The class split is moderately imbalanced, so F1-score and ROC-AUC are better optimization targets than raw accuracy. This supports using balanced model settings and threshold-aware evaluation.

## 2) Engagement vs Churn

We test whether low engagement users churn more frequently.

In [ ]:
activity_band = pd.cut(
    abt_df["total_seconds_active"],
    bins=[-1, 300, 500, float("inf")],
    labels=["<300s", "300-500s", ">500s"],
)

engagement_churn = (
    abt_df.assign(activity_band=activity_band)
    .groupby("activity_band", observed=False)["churned"]
    .mean()
    .reset_index(name="churn_rate")
)

plt.figure(figsize=(7, 4.5))
sns.barplot(data=engagement_churn, x="activity_band", y="churn_rate", hue="activity_band", dodge=False, legend=False)
plt.title("Churn Rate by Engagement Band")
plt.xlabel("Engagement Band")
plt.ylabel("Churn Rate")
plt.tight_layout()
plt.show()

print(engagement_churn)

### Takeaway

Lower engagement bands show materially higher churn rates. This indicates early engagement campaigns (especially for low-activity users) are likely high-impact interventions.

## 3) Correlation Scan (Numeric Features)

We inspect coarse linear relationships to identify candidate predictive signals and redundancy.

In [ ]:
numeric_cols = abt_df.select_dtypes(include=["number"]).columns.tolist()
keep = [c for c in numeric_cols if c != "customer_id"]
corr = abt_df[keep].corr(numeric_only=True)

plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.3)
plt.title("Numeric Correlation Heatmap")
plt.tight_layout()
plt.show()

corr_with_target = corr["churned"].sort_values(ascending=False)
print("Top positive correlations with churned:")
print(corr_with_target.head(10))
print("\nTop negative correlations with churned:")
print(corr_with_target.tail(10))

### Takeaway

Recency and usage-intensity features consistently show strong churn relationships, supporting their priority in feature engineering and retention policy design. Correlation is not causation, but this gives a practical directional signal for model iteration.